# Kori Hugging Face SFT (Google Colab)

This notebook fine-tunes a chat model with LoRA/QLoRA using your `text` JSONL file and pushes outputs to Hugging Face Hub.

Recommended runtime: **GPU (T4 or better)** in Google Colab.

In [ ]:
!pip -q install --upgrade \"transformers>=4.46.0\" \"datasets>=3.0.0\" \"trl>=0.11.0\" \"peft>=0.13.0\" \"accelerate>=1.0.1\" \"bitsandbytes>=0.44.1\" \"huggingface_hub>=0.26.0\"

In [ ]:
import os
import torch
from google.colab import files
from datasets import load_dataset
from huggingface_hub import HfApi
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from trl import SFTTrainer

HF_TOKEN = ""  # paste your token
BASE_MODEL = "meta-llama/Llama-3.2-1B-Instruct"
OUTPUT_REPO = ""  # optional: set explicitly, e.g. NKINYAM/kori-llama32-1b-sft
ADAPTER_REPO = ""  # optional adapter repo, e.g. NKINYAM/kori-llama32-1b-sft-lora

NUM_EPOCHS = 1
BATCH_SIZE = 2
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4
MAX_SEQ_LENGTH = 1024

assert HF_TOKEN, "Set HF_TOKEN first"
os.environ["HF_TOKEN"] = HF_TOKEN
api = HfApi(token=HF_TOKEN)
username = api.whoami().get("name", "").strip()
assert username, "Could not resolve Hugging Face username from token"

if not OUTPUT_REPO:
    OUTPUT_REPO = f"{username}/kori-llama32-1b-sft"
if not ADAPTER_REPO:
    ADAPTER_REPO = f"{username}/kori-llama32-1b-sft-lora"

print("HF user:", username)
print("Base model:", BASE_MODEL)
print("Adapter repo:", ADAPTER_REPO)
print("Merged model repo:", OUTPUT_REPO)

In [ ]:
uploaded = files.upload()
train_candidates = [name for name in uploaded.keys() if name.endswith(".jsonl")]
assert train_candidates, "Upload your hf_sft_train_*.jsonl file"
TRAIN_FILE = f"/content/{train_candidates[0]}"
print("Training file:", TRAIN_FILE)

In [ ]:
dataset = load_dataset("json", data_files={"train": TRAIN_FILE})["train"]
assert len(dataset) > 0, "Dataset is empty"
assert "text" in dataset.column_names, "Dataset must contain a text column"
print("Rows:", len(dataset))
print(dataset[0]["text"][:400])

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)

training_args = TrainingArguments(
    output_dir="/content/kori_sft_out",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,
    bf16=False,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    tokenizer=tokenizer,
    args=training_args,
    packing=True,
)

trainer.train()

In [ ]:
ADAPTER_LOCAL_DIR = "/content/kori_adapter"
trainer.model.save_pretrained(ADAPTER_LOCAL_DIR)
tokenizer.save_pretrained(ADAPTER_LOCAL_DIR)

api.create_repo(repo_id=ADAPTER_REPO, repo_type="model", private=False, exist_ok=True)
api.upload_folder(repo_id=ADAPTER_REPO, repo_type="model", folder_path=ADAPTER_LOCAL_DIR)
print("Adapter uploaded:", ADAPTER_REPO)

In [ ]:
# Merge LoRA into a full model so it is easier to consume in production.
from peft import AutoPeftModelForCausalLM

merge_model = AutoPeftModelForCausalLM.from_pretrained(
    ADAPTER_LOCAL_DIR,
    torch_dtype=torch.float16,
    device_map="auto",
)
merge_model = merge_model.merge_and_unload()

MERGED_LOCAL_DIR = "/content/kori_merged_model"
merge_model.save_pretrained(MERGED_LOCAL_DIR, safe_serialization=True, max_shard_size="2GB")
tokenizer.save_pretrained(MERGED_LOCAL_DIR)

api.create_repo(repo_id=OUTPUT_REPO, repo_type="model", private=False, exist_ok=True)
api.upload_folder(repo_id=OUTPUT_REPO, repo_type="model", folder_path=MERGED_LOCAL_DIR)
print("Merged model uploaded:", OUTPUT_REPO)

## Optional: 6 Questions / 24h Limiter (Notebook Demo)

Use this helper during notebook-side testing to enforce per-user usage limits.
It allows up to **6 questions** in a rolling **24-hour** window.

In [ ]:
from datetime import datetime, timedelta, timezone

QUESTION_LIMIT = 6
WINDOW_HOURS = 24
usage_counter = {}

def check_and_increment(user_id: str):
    now = datetime.now(timezone.utc)
    record = usage_counter.get(user_id)

    if not record or now - record["window_start"] >= timedelta(hours=WINDOW_HOURS):
        record = {"count": 0, "window_start": now}

    if record["count"] >= QUESTION_LIMIT:
        retry_at = record["window_start"] + timedelta(hours=WINDOW_HOURS)
        usage_counter[user_id] = record
        return False, f"You have reached your {QUESTION_LIMIT}-question limit. Please continue tomorrow (after {retry_at.isoformat()}).", record["count"]

    record["count"] += 1
    usage_counter[user_id] = record
    return True, "Allowed. Answer normally.", record["count"]

# Example
ok, message, count = check_and_increment("demo-user-1")
print({"ok": ok, "count": count, "message": message})

## Use In This App

After upload completes, set these in your project `.env`:

```bash
AI_PROVIDER="huggingface"
HUGGING_FACE_API_KEY="<your-hf-token>"
HUGGING_FACE_MODEL="<your-merged-model-repo-id>"
AI_CHAT_DAILY_LIMIT="6"
AI_CHAT_LIMIT_WINDOW_MS="86400000"
```

Then restart your backend server.